### Ajuste hiperparámetros usando TreeDecisionClassifier

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
#Función que recibe un dataset y realiza la validación one leave out 
# para devolver los parámetros óptimos del modelo TreeDecisionClassifier
# Parámtros a optimizar:
# - Número máximo de hojas del árbol
# - Profundidad máxima del árbol
# - Número mínimo de ejemplos para hacer unaa partición
# Se utilizará la métrica de precisión para evaluar el rendimiento del modelo

def preprocesado(df, label='Label'):
    # Conversión de las variables categóricas a numéricas
    df.replace({
        "Age": {"young": 0, "pre-presbyopic": 1, "presbyopic": 2},
        "Prescription": {"myope": 0, "hypermetrope": 1},
        "Astigmatic": {"no": 0, "yes": 1},
        "TearRate": {"reduced": 0, "normal": 1},
        "Label": {"no lenses": 0, "soft": 1, "hard": 2}
    }, inplace=True)
    
    # Separación de las características (X) y la variable objetivo (y)
    X = df.drop(columns=[label])
    #Converisón a variables binarias (one hot encoding)
    X = pd.get_dummies(X, columns=X.columns)

    y = df[label]
    return X.values, y.values

def optimizar_arbol_decision(dataset, label='Label'):

    X,y = preprocesado(dataset, label=label)
    # Definir los rangos de los parámetros a optimizar
    max_leaf_nodes_range = [5, 10, 20, 50]
    max_depth_range = [3, 5, 10, None]
    min_samples_split_range = [2, 5, 10]

    best_precision = 0
    best_params = {}

    # Iterar sobre todas las combinaciones de parámetros
    for max_leaf_nodes in max_leaf_nodes_range:
        for max_depth in max_depth_range:
            for min_samples_split in min_samples_split_range:
                # Crear el modelo con los parámetros actuales
                model = DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes,
                                               max_depth=max_depth,
                                               min_samples_split=min_samples_split)

                # Validación one leave out
                loo = LeaveOneOut()
                precisions = []

                for train_index, test_index in loo.split(X):
                    X_train, X_test = X[train_index], X[test_index]
                    y_train, y_test = y[train_index], y[test_index]

                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                    precisions.append(precision_score(y_test, y_pred))

                # Calcular la precisión promedio para esta combinación de parámetros
                avg_precision = np.mean(precisions)

                # Actualizar los mejores parámetros si la precisión es mejor
                if avg_precision > best_precision:
                    best_precision = avg_precision
                    best_params = {
                        'max_leaf_nodes': max_leaf_nodes,
                        'max_depth': max_depth,
                        'min_samples_split': min_samples_split
                    }

    return best_params



In [ ]:
df_train = pd.read_csv('data/lenses.csv')

#plt.figure(figsize=(12,6))
#sns.countplot(x='Label', data=df_train, hue="Label")

X,y = preprocesado(df_train)
print(X[:5])
print(y[:5])

   Age_0  Age_1  Age_2  Prescription_0  Prescription_1  Astigmatic_0  \
0   True  False  False            True           False          True   
1   True  False  False            True           False          True   
2   True  False  False            True           False         False   
3   True  False  False            True           False         False   
4   True  False  False           False            True          True   

   Astigmatic_1  TearRate_0  TearRate_1  
0         False        True       False  
1         False       False        True  
2          True        True       False  
3          True       False        True  
4         False        True       False  
0    0
1    1
2    0
3    2
4    0
Name: Label, dtype: int64
